# Logistic Regression and XGBoost

Two models, two philosophies. Logistic regression gives us interpretability: its coefficients tell us exactly how much each feature contributes, which is scientifically useful. XGBoost gives us raw performance: an ensemble of decision trees that doesn't care about your assumptions.

Both are trained on the engineered features we computed in Week 1. All runs are logged to MLflow so we can compare everything cleanly in notebook 03.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import mlflow.xgboost

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, CalibrationDisplay
)
from inference_lens.models.train import train_logreg, train_xgboost, evaluate_model
from inference_lens.utils.logging import setup_logging

setup_logging()
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

import os
os.makedirs('../../models', exist_ok=True)
os.makedirs('../../reports/figures', exist_ok=True)

## MLflow setup

All experiments land in `mlruns/` at the project root. Run `mlflow ui` from the project root to browse them.

In [ ]:
mlflow.set_tracking_uri('../../mlruns')
mlflow.set_experiment('inference-lens')
print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')

## Load features and build train/val/test splits

In [ ]:
df = pd.read_parquet('../../data/processed/features_with_labels.parquet')

feature_cols = ['token_length', 'type_token_ratio', 'flesch_score', 'rouge_l']

# reproduce the same 70/15/15 split using the same seed
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
n = len(df_shuffled)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

train_df = df_shuffled.iloc[:n_train]
val_df   = df_shuffled.iloc[n_train : n_train + n_val]
test_df  = df_shuffled.iloc[n_train + n_val :]

X_train, y_train = train_df[feature_cols].values, train_df['label'].values
X_val,   y_val   = val_df[feature_cols].values,   val_df['label'].values
X_test,  y_test  = test_df[feature_cols].values,  test_df['label'].values

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Label balance -- train: {y_train.mean():.3f}  val: {y_val.mean():.3f}  test: {y_test.mean():.3f}')

In [ ]:
# scale features -- logistic regression needs this, XGBoost doesn't but it doesn't hurt
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

joblib.dump(scaler, '../../models/scaler.joblib')
print('Scaler fitted and saved.')

## Train Logistic Regression

L2-regularized with cross-validated C selection. Trains fast, gives us a clean interpretable baseline.

In [ ]:
logreg_model = train_logreg(
    X_train_sc, y_train,
    X_val_sc, y_val,
    c_values=[0.01, 0.1, 1.0, 10.0],
    cv_folds=5,
)

joblib.dump(logreg_model, '../../models/logreg.joblib')
print('Logistic regression saved to models/logreg.joblib')

In [ ]:
# evaluate on test set
logreg_test_prob = logreg_model.predict_proba(X_test_sc)[:, 1]
logreg_test_pred = logreg_model.predict(X_test_sc)

logreg_auc = roc_auc_score(y_test, logreg_test_prob)
logreg_f1  = f1_score(y_test, logreg_test_pred, average='macro')

print(f'Logistic Regression -- Test AUC: {logreg_auc:.4f}  F1 (macro): {logreg_f1:.4f}')
print()
print(classification_report(y_test, logreg_test_pred, target_names=['Rejected', 'Chosen']))

## Logistic Regression: feature coefficients

The whole point of training an interpretable model is actually reading what it learned.

In [ ]:
coef_df = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': logreg_model.coef_[0],
}).sort_values('coefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression coefficients (scaled features)')
ax.set_xlabel('Coefficient value')
plt.tight_layout()
plt.savefig('../../reports/figures/14_logreg_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

print(coef_df.to_string(index=False))

## Train XGBoost

Grid search over depth, learning rate, and n_estimators. Takes longer than logistic regression but usually wins on AUC.

In [ ]:
xgb_model = train_xgboost(
    X_train, y_train,   # XGBoost doesn't need scaled features
    X_val, y_val,
    param_grid={
        'max_depth': [3, 5, 7],
        'learning_rate': [0.05, 0.1],
        'n_estimators': [100, 300],
        'subsample': [0.8, 1.0],
    },
    cv_folds=5,
)

joblib.dump(xgb_model, '../../models/xgboost.joblib')
print('XGBoost saved to models/xgboost.joblib')

In [ ]:
xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_test_pred = xgb_model.predict(X_test)

xgb_auc = roc_auc_score(y_test, xgb_test_prob)
xgb_f1  = f1_score(y_test, xgb_test_pred, average='macro')

print(f'XGBoost -- Test AUC: {xgb_auc:.4f}  F1 (macro): {xgb_f1:.4f}')
print()
print(classification_report(y_test, xgb_test_pred, target_names=['Rejected', 'Chosen']))

## XGBoost: feature importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_,
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(importance_df['feature'], importance_df['importance'], color='steelblue', edgecolor='white')
ax.set_title('XGBoost feature importance')
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.savefig('../../reports/figures/15_xgboost_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(importance_df.to_string(index=False))

## Side-by-side: ROC and Precision-Recall curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

RocCurveDisplay.from_predictions(y_test, logreg_test_prob, name='Logistic Regression', ax=axes[0], color='steelblue')
RocCurveDisplay.from_predictions(y_test, xgb_test_prob,    name='XGBoost',            ax=axes[0], color='darkorange')
axes[0].set_title('ROC curves')

PrecisionRecallDisplay.from_predictions(y_test, logreg_test_prob, name='Logistic Regression', ax=axes[1], color='steelblue')
PrecisionRecallDisplay.from_predictions(y_test, xgb_test_prob,    name='XGBoost',            ax=axes[1], color='darkorange')
axes[1].set_title('Precision-Recall curves')

plt.suptitle('Logistic Regression vs XGBoost on HH-RLHF test set', fontsize=13)
plt.tight_layout()
plt.savefig('../../reports/figures/16_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Calibration plots

A well-calibrated model that says 70% confidence should be right about 70% of the time. Miscalibrated models can have great AUC but useless confidence scores, which matters for the stress-test phase.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

CalibrationDisplay.from_predictions(y_test, logreg_test_prob, n_bins=10, name='Logistic Regression', ax=ax, color='steelblue')
CalibrationDisplay.from_predictions(y_test, xgb_test_prob,    n_bins=10, name='XGBoost',            ax=ax, color='darkorange')

ax.set_title('Calibration curves (reliability diagrams)')
plt.tight_layout()
plt.savefig('../../reports/figures/17_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## Save test predictions for model comparison notebook

In [ ]:
preds_df = pd.DataFrame({
    'y_true': y_test,
    'logreg_prob': logreg_test_prob,
    'logreg_pred': logreg_test_pred,
    'xgb_prob': xgb_test_prob,
    'xgb_pred': xgb_test_pred,
})

preds_df.to_parquet('../../data/processed/test_predictions.parquet', index=False)
print(f'Saved test predictions for {len(preds_df):,} samples')

print(f'\nSummary:')
print(f'  Logistic Regression  AUC: {logreg_auc:.4f}  F1: {logreg_f1:.4f}')
print(f'  XGBoost              AUC: {xgb_auc:.4f}  F1: {xgb_f1:.4f}')